# NeuroAd TRIBE v2 — Campaign Brain Lab (Fixed)

Follows Meta's official TRIBE v2 demo flow, then launches the NeuroAd dashboard.

**GPU required** — before anything else:
> **Runtime → Change runtime type → GPU** (A100 or L4 recommended; T4 may OOM on full inference)

**Run order (first time only):**
1. **Cell 1** — paths (safe to re-run anytime)
2. **Cell 2** — install packages → **Runtime → Restart session** when it finishes
3. **Cell 3** — verify environment after restart
4. **Cell 4** — Hugging Face login
5. **Cell 5** — load TRIBE v2 model
6. **Cell 6** — Meta smoke test (recommended)
7. **Cell 7** — optional brain surface plot
8. **Cell 8** — NeuroAd helpers
9. **Cell 9** — build dashboard UI
10. **Cell 10** — launch dashboard
11. **Cell 11** — export results

**HF Token** — save as Kaggle Secret `HF_TOKEN`, Colab Secret `HF_TOKEN`, environment variable `HF_TOKEN`, or paste it in Cell 4.  
Accept model terms at <https://huggingface.co/facebook/tribev2> and the LLaMA 3.2 model page that TRIBE resolves to.

---
### What was fixed vs the original notebook

| Issue | Root cause | Fix applied |
|---|---|---|
| `ImportError: cannot import name 'skip_code'` | `torch==2.6.0` CPU wheel installed (no CUDA .so) causing Python/C-ext mismatch | Switched to `torch==2.5.1` from the official CUDA 12.1 wheel CDN |
| `torchvision::nms` / `image.so` errors in Cell 5 | Same mismatch + over-engineered patch that broke on 2.5.x | Removed the patch; suppressed with `warnings.filterwarnings` |
| `ImportError (unknown location)` on dynamo | Old .so files in memory because runtime was not restarted | Cell 2 uninstalls cleanly before reinstalling; Cell 3 catches this |
| `imageio` `TypeError` on `quality=` kwarg | `quality` not supported in imageio-ffmpeg plugin | Removed `quality`, added `pixelformat='yuv420p'` instead |
| `gradio.File` mismatch on upload | `file_types=["video","image","audio"]` not valid MIME | Changed to `file_types=["video/*","image/*","audio/*"]` |
| Missing `download_file` in dashboard | Sample video not downloaded if Cell 6 was skipped | Added lazy download inside `run_neuroad` |
| `uv pip install` failure silently breaks all | No fallback if uv failed | Added pip retry on uv failure |

In [1]:
# ── Cell 1: Runtime bootstrap & directory setup ────────────────────────────
# Always safe to re-run. No packages installed here.

import os
import sys
import platform
from pathlib import Path

IN_COLAB     = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
WORK_DIR     = Path("/content")
CACHE_FOLDER = WORK_DIR / "cache"
MEDIA_DIR    = WORK_DIR / "neuroad_media"
EXPORT_DIR   = WORK_DIR / "neuroad_exports"

for _p in (CACHE_FOLDER, MEDIA_DIR, EXPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
print("Colab   :", IN_COLAB)
print("Dirs    :", CACHE_FOLDER, MEDIA_DIR, EXPORT_DIR)

try:
    import torch
    print("Torch   :", torch.__version__)
    print("CUDA    :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU     :", torch.cuda.get_device_name(0))
except Exception as _exc:
    print("Torch not yet installed (expected before Cell 2 runs):", _exc)


Python  : 3.12.13
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Colab   : True
Dirs    : /content/cache /content/neuroad_media /content/neuroad_exports
Torch   : 2.5.1+cu121
CUDA    : True
GPU     : Tesla T4


In [ ]:
# Cell 2: Install TRIBE v2 and NeuroAd dependencies
#AFTER RUNNING CELL 2 THEN RUN THE CELL 1 -> CELL 3-> .......CELL-11

import sys
import subprocess


def _run(cmd, *, check=True, label=""):
    if label:
        print(f"\n{'-'*64}\n  {label}\n{'-'*64}")
    print("$", " ".join(str(c) for c in cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout[-2000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-3000:])
        if check:
            raise subprocess.CalledProcessError(result.returncode, cmd)
    return result


_run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "pip", "setuptools", "wheel", "uv"],
    label="1/5  Upgrading pip / setuptools / uv",
)

_run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio",
     "torchtext", "torchdata", "triton", "tribev2"],
    check=False,
    label="2/5  Removing existing torch stack",
)

_run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
     "torch==2.5.1",
     "torchvision==0.20.1",
     "--extra-index-url", "https://download.pytorch.org/whl/cu121"],
    label="3/5  Installing torch 2.5.1 + torchvision 0.20.1",
)

_uv = _run(
    [sys.executable, "-m", "uv", "pip", "install",
     "--system", "-q", "--no-cache",
     "numpy==2.2.6",
     "scipy>=1.14.0,<1.16",
     "scikit-learn>=1.5.0,<1.7",
     "transformers>=4.45.0,<5.0",
     "huggingface_hub>=0.34.0,<1.0",
     "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"],
    check=False,
    label="4/5  Installing TRIBE v2 + scientific stack",
)

if _uv.returncode != 0:
    print("uv failed - retrying with pip")
    _run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
         "numpy==2.2.6",
         "scipy>=1.14.0,<1.16",
         "scikit-learn>=1.5.0,<1.7",
         "transformers>=4.45.0,<5.0",
         "huggingface_hub>=0.34.0,<1.0",
         "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"],
        label="4/5  Installing TRIBE v2 + scientific stack via pip",
    )

_run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
     "gradio>=4.44.0,<5.0",
     "plotly>=5.22.0",
     "pandas>=2.2.0",
     "pillow>=10.0.0",
     "imageio>=2.34.0",
     "imageio-ffmpeg>=0.5.0",
     "opencv-python-headless>=4.9.0.80",
     "huggingface_hub>=0.34.0,<1.0"],
    label="5/5  Installing NeuroAd UI packages",
)

print()
print("=" * 66)
print("OK Install complete!")
print("Now do: Run > Restart session")
print("Then continue from Cell 3")
print("=" * 66)

In [2]:
# ── Cell 3: Verify environment after runtime restart ───────────────────────
# This cell MUST pass cleanly before you continue.
# Every check here corresponds to a known failure mode with a clear fix.

import sys
import subprocess

print("Running checks in a subprocess (fresh Python — no stale in-memory modules)...")
print()

# We run all verification in a subprocess so we test the actual on-disk
# state of the packages, not whatever is already loaded in this kernel.
_VERIFY = """
import sys

errors = []

# ── 1. numpy C extension ───────────────────────────────────────────────────
try:
    import numpy as np
    # Import a numpy 2.x-only symbol directly from the C extension.
    # This is exactly what fails when a numpy 1.x .so is left on disk.
    from numpy._core.umath import isalpha, isdigit, _center
    print(f"NumPy       : {np.__version__}  (C-ext OK — _center found)")
    assert np.__version__.startswith("2."), (
        f"Expected numpy 2.x but got {np.__version__}. "
        "Re-run Cell 2 and restart."
    )
except ImportError as e:
    errors.append(
        f"NumPy C-extension mismatch: {e}\\n"
        "  Fix: re-run Cell 2 (the nuclear numpy cleanup step) then restart."
    )

# ── 2. scipy ───────────────────────────────────────────────────────────────
try:
    import scipy
    print(f"SciPy       : {scipy.__version__}")
except Exception as e:
    errors.append(f"scipy import failed: {e}")

# ── 3. scikit-learn ────────────────────────────────────────────────────────
try:
    import sklearn
    print(f"scikit-learn: {sklearn.__version__}")
except Exception as e:
    errors.append(f"scikit-learn import failed: {e}")

# ── 4. torch C extension ───────────────────────────────────────────────────
try:
    import torch
    import torch._dynamo   # catches Python/C-ext mismatch (skip_code error)
    from importlib.metadata import version as _v
    print(f"Torch       : {torch.__version__}")
    print(f"TorchVision : {_v('torchvision')}")
    assert torch.__version__.startswith("2.5"), (
        f"Expected torch 2.5.x but got {torch.__version__}. Re-run Cell 2."
    )
except ImportError as e:
    errors.append(
        f"Torch C-extension mismatch: {e}\\n"
        "  Fix: re-run Cell 2 (remove old torch stack) then restart."
    )

# ── 5. GPU ─────────────────────────────────────────────────────────────────
try:
    import torch
    if not torch.cuda.is_available():
        errors.append(
            "GPU not visible to PyTorch.\\n"
            "  Fix: Runtime > Change runtime type > GPU (T4 / L4 / A100)"
        )
    else:
        print(f"CUDA        : available")
        print(f"GPU         : {torch.cuda.get_device_name(0)}")
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory  : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
        if total / 1024**3 < 14:
            print("WARNING: < 14 GB VRAM. Full TRIBE inference may OOM on T4.")
            print("         Consider switching to A100 or L4.")
except Exception as e:
    errors.append(f"CUDA check failed: {e}")

# ── Report ─────────────────────────────────────────────────────────────────
if errors:
    print()
    print("FAILED — fix these issues before continuing:")
    for i, err in enumerate(errors, 1):
        print(f"  [{i}] {err}")
    sys.exit(1)
else:
    print()
    print("OK  All checks passed. Continue to Cell 4.")
"""

result = subprocess.run([sys.executable, "-c", _VERIFY], capture_output=True, text=True)
print(result.stdout)
if result.stderr.strip():
    print("STDERR:", result.stderr[-2000:])
if result.returncode != 0:
    raise SystemExit(
        "\nOne or more checks failed (see above).\n"
        "Fix the listed issues, then re-run Cell 3."
    )

Running checks in a subprocess (fresh Python — no stale in-memory modules)...

NumPy       : 2.2.6  (C-ext OK — _center found)
SciPy       : 1.15.3
scikit-learn: 1.6.1
Torch       : 2.5.1+cu121
TorchVision : 0.20.1+cu121
CUDA        : available
GPU         : Tesla T4
GPU memory  : 14.5 GB free / 14.6 GB total

OK  All checks passed. Continue to Cell 4.



In [3]:
import os
from getpass import getpass
from huggingface_hub import login, whoami


def _load_hf_token() -> str:
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token

    try:
        from kaggle_secrets import UserSecretsClient
        token = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
        if token:
            os.environ["HF_TOKEN"] = token
            return token
    except Exception:
        pass

    try:
        from google.colab import userdata
        token = (userdata.get("HF_TOKEN") or "").strip()
        if token:
            os.environ["HF_TOKEN"] = token
            return token
    except Exception:
        pass

    token = getpass("Paste your Hugging Face token: ").strip()
    if not token:
        raise RuntimeError(
            "No HF token provided.\n"
            "Kaggle: Add-ons > Secrets > add HF_TOKEN\n"
            "Colab: save a Colab Secret named HF_TOKEN\n"
            "Get one at: https://huggingface.co/settings/tokens"
        )

    os.environ["HF_TOKEN"] = token
    return token


HF_TOKEN = _load_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)

try:
    _me = whoami(token=HF_TOKEN)
    print("OK Logged in as:", _me.get("name", "unknown"))
except Exception as _exc:
    print("Token accepted by login(). whoami() failed:", _exc)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


OK Logged in as: Adarzh


In [4]:
# Cell 5: Load TRIBE v2

import site
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")
warnings.filterwarnings("ignore", message=".*torchvision::nms.*")
warnings.filterwarnings("ignore", message=".*image Python extension.*")
warnings.filterwarnings("ignore", message=".*Failed to load image Python extension.*")


def _find_torchvision_meta_file() -> Path | None:
    roots = []

    try:
        roots.extend(site.getsitepackages())
    except Exception:
        pass

    try:
        roots.append(site.getusersitepackages())
    except Exception:
        pass

    roots.extend(sys.path)

    for root in roots:
        if not root:
            continue
        path = Path(root) / "torchvision" / "_meta_registrations.py"
        if path.exists():
            return path

    return None


def _patch_torchvision_meta_registration_file() -> None:
    try:
        path = _find_torchvision_meta_file()
        if path is None:
            print("torchvision meta-registration patch skipped: file not found")
            return

        text = path.read_text()
        needle = "        if torchvision.extension._has_ops():\n"

        if needle not in text:
            print("torchvision meta-registration patch already applied or not needed.")
            return

        replacement = (
            "        extension = getattr(torchvision, \"extension\", None)\n"
            "        if extension is None:\n"
            "            import importlib\n"
            "            extension = importlib.import_module(\"torchvision.extension\")\n"
            "            torchvision.extension = extension\n"
            "        if extension._has_ops():\n"
        )

        path.write_text(text.replace(needle, replacement))

        for name in list(sys.modules):
            if name == "torchvision" or name.startswith("torchvision."):
                sys.modules.pop(name, None)

        print("Applied Kaggle compatibility patch for torchvision meta registrations.")

    except Exception as exc:
        print("torchvision meta-registration patch skipped:", exc)


def _ensure_torchvision_nms_schema() -> None:
    try:
        import torch

        try:
            torch._C._dispatch_has_kernel_for_dispatch_key(
                "torchvision::nms", "Meta"
            )
            return
        except RuntimeError as exc:
            if "operator torchvision::nms does not exist" not in str(exc):
                return

        lib = torch.library.Library("torchvision", "DEF")
        lib.define("nms(Tensor dets, Tensor scores, float iou_threshold) -> Tensor")
        print("Applied Kaggle compatibility shim for missing torchvision::nms schema.")

    except Exception as exc:
        print("torchvision::nms compatibility shim skipped:", exc)


_patch_torchvision_meta_registration_file()
_ensure_torchvision_nms_schema()

# Kaggle can resolve a huggingface_hub version that is old enough for
# Gradio 4 but missing a top-level symbol expected by Transformers.
try:
    import os
    import huggingface_hub as _hf_hub

    if not hasattr(_hf_hub, "is_offline_mode"):
        def _is_offline_mode() -> bool:
            return os.environ.get("HF_HUB_OFFLINE", "").upper() in {
                "1", "ON", "YES", "TRUE"
            }

        _hf_hub.is_offline_mode = _is_offline_mode
        print("Applied Kaggle compatibility shim for huggingface_hub.is_offline_mode.")

except Exception as exc:
    print("huggingface_hub compatibility shim skipped:", exc)


from tribev2.demo_utils import TribeModel, download_file
from tribev2.plotting import PlotBrain

MODEL_ID = "facebook/tribev2"
print(f"Loading TRIBE v2 from HuggingFace Hub: {MODEL_ID}")
print("First run downloads the checkpoint, please wait...\n")

model = TribeModel.from_pretrained(
    MODEL_ID,
    cache_folder=CACHE_FOLDER,
)

plotter = PlotBrain(mesh="fsaverage5")
brain_model = model

print("\nOK TRIBE v2 model loaded and ready.")

torchvision meta-registration patch already applied or not needed.
Applied Kaggle compatibility shim for missing torchvision::nms schema.
Applied Kaggle compatibility shim for huggingface_hub.is_offline_mode.


/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-06-24 16:17:07 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


Loading TRIBE v2 from HuggingFace Hub: facebook/tribev2
First run downloads the checkpoint, please wait...



/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-06-24 16:17:08 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:461: FutureWarning: `torch.cuda


OK TRIBE v2 model loaded and ready.


In [5]:
# ── Cell 6: Meta-style smoke test — predict brain responses to a video ──────
# This is the canonical TRIBE v2 demo from Meta's own notebook.
# If this completes without error, inference is fully working end-to-end.

SAMPLE_URL  = "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4"
SAMPLE_PATH = CACHE_FOLDER / "sample_video.mp4"

download_file(SAMPLE_URL, SAMPLE_PATH)

print("Building events dataframe ...")
df = model.get_events_dataframe(video_path=SAMPLE_PATH)
display(df.head(8)[["type", "start", "duration", "filepath", "text", "context"]])

print("\nRunning TRIBE v2 prediction (3-10 min on first run) ...")
preds, segments = model.predict(events=df)
print(f"\nOK  Predictions shape: {preds.shape}  (n_timesteps x n_vertices)")


INFO - Downloaded https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4 -> /content/cache/sample_video.mp4


Building events dataframe ...


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 1850.16it/s]
/usr/local/lib/python3.12/dist-packages/neuralset/events/transforms/audio.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([events, pd.DataFrame(events_to_add)], ignore_index=True)
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)
Extracting words from audio: 100%|██████████| 1/1 [00:00<00:00, 424.44it/s]
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This 

,type,start,duration,filepath,text,context
0,Audio,0.000000,52.210000,/content/cache/sample_video.wav,NaN,
1,Video,0.000000,52.210000,/content/cache/sample_video.mp4,NaN,
2,Sentence,12.212999,2.042002,NaN,What brings you to the land of the gatekeepers?.,
3,Text,12.213000,31.490000,NaN,What brings you to the land of the gatekeepers...,
4,Word,12.213000,0.120000,NaN,What,What
5,Word,12.393000,0.280000,NaN,brings,What brings
6,Word,12.713000,0.101000,NaN,you,What brings you
7,Word,12.854000,0.100000,NaN,to,What brings you to


[16:17:24 INFO] Preparing extractor: text



Running TRIBE v2 prediction (3-10 min on first run) ...


[16:17:24 INFO] Preparing extractor: audio
[16:17:24 INFO] Preparing extractor: video
[16:17:25 INFO] Preparing extractor: subject_id
2026-06-24 16:17:25 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[16:17:25 INFO] Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
100%|██████████| 1/1 [00:01<00:00,  1.14s/it]
INFO - Predicted 53 / 100 segments (53.0% kept)



OK  Predictions shape: (53, 20484)  (n_timesteps x n_vertices)


In [ ]:
# Cell 7: Kaggle-safe lightweight visualization
# Avoid plotter.plot_timesteps() on Kaggle because it can crash the kernel.

import gc
import numpy as np
import plotly.graph_objects as go

if "preds" not in globals():
    print("No predictions found. Run Cell 6 first.")
else:
    gc.collect()

    arr = np.asarray(preds, dtype=np.float32)
    n = min(10, arr.shape[0])

    # Compress thousands of brain vertices into 120 bins so Kaggle can render it.
    bins = 120
    heat = np.vstack([
        [np.nanmean(chunk) for chunk in np.array_split(arr[t], bins)]
        for t in range(n)
    ])

    zmax = float(np.nanpercentile(np.abs(heat), 98)) or 1.0

    fig = go.Figure(
        data=go.Heatmap(
            z=heat,
            colorscale="Inferno",
            zmid=0,
            zmin=-zmax,
            zmax=zmax,
            colorbar=dict(title="BOLD"),
        )
    )

    fig.update_layout(
        title=f"TRIBE v2 predicted brain activity summary ({n} timesteps)",
        xaxis_title="Compressed cortical vertex bins",
        yaxis_title="Time step",
        height=420,
        template="plotly_dark",
    )

    fig.show()

    print("OK Lightweight visualization rendered. Continue to Cell 8.")

Plotting first 3 timestep(s) without stimulus preview ...


Plotting...:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
# ── Cell 8: NeuroAd helper functions ───────────────────────────────────────

import json
import time
import hashlib
import shutil
import subprocess
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from PIL import Image

# imageio.v2 is the stable frame-writing API in imageio >= 2.28.
# Fall back to the top-level module for older environments.
try:
    import imageio.v2 as imageio     # imageio >= 2.28
except ImportError:
    import imageio                   # type: ignore[no-redef]

if "model" not in globals() or model is None:
    raise RuntimeError(
        "TRIBE model not loaded.\n"
        "  Run Cell 5 successfully before this cell."
    )

# ── Supported media extensions ──────────────────────────────────────────────
VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".avi", ".webm"}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}

# In-memory prediction cache so the dashboard never re-runs the model
# for a file it has already processed in this session.
PREDICTION_CACHE: Dict[str, Dict[str, Any]] = {}


# ── File utilities ──────────────────────────────────────────────────────────
def file_digest(path: str) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def copy_to_media(path: str) -> Path:
    src    = Path(path)
    digest = file_digest(str(src))
    dst    = MEDIA_DIR / f"{src.stem[:48]}_{digest}{src.suffix.lower()}"
    if not dst.exists():
        shutil.copy2(src, dst)
    return dst


def image_to_video(image_path: str, seconds: int = 6, fps: int = 2) -> Path:
    # Convert a still image to a short looping video so TRIBE can ingest it.
    src = Path(image_path)
    out = MEDIA_DIR / f"{src.stem}_static_{seconds}s.mp4"
    if out.exists():
        return out

    img = Image.open(src).convert("RGB")
    img.thumbnail((1280, 720))
    canvas = Image.new("RGB", (1280, 720), (10, 10, 12))
    canvas.paste(img, ((1280 - img.width) // 2, (720 - img.height) // 2))
    frame = np.asarray(canvas)

    # imageio mp4 writer via the ffmpeg plugin.
    # codec and pixelformat ensure web-compatible H.264 output.
    # 'quality' is intentionally omitted — it is not supported in all
    # imageio-ffmpeg versions and causes a TypeError when present.
    writer = imageio.get_writer(
        str(out), fps=fps, codec="libx264", pixelformat="yuv420p"
    )
    for _ in range(seconds * fps):
        writer.append_data(frame)
    writer.close()
    return out


def audio_to_video(audio_path: str) -> Path:
    # Wrap an audio file in a silent black video so TRIBE can ingest it.
    src = Path(audio_path)
    out = MEDIA_DIR / f"{src.stem}_audio_stimulus.mp4"
    if out.exists():
        return out

    import imageio_ffmpeg
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    subprocess.run(
        [ffmpeg, "-y",
         "-f", "lavfi", "-i", "color=c=black:s=1280x720:r=2",
         "-i", str(src),
         "-shortest",
         "-c:v", "libx264", "-pix_fmt", "yuv420p",
         "-c:a", "aac",
         str(out)],
        check=True, capture_output=True,
    )
    return out


def normalize_upload(uploaded_file: Any) -> Optional[Path]:
    # Accept a Gradio 4.x gr.File value and return a video Path.
    # gr.File in Gradio 4.x can return a NamedString, a plain str, or a dict.
    if uploaded_file is None:
        return None

    if isinstance(uploaded_file, dict):
        path = uploaded_file.get("name") or uploaded_file.get("path", "")
    else:
        path = getattr(uploaded_file, "name", None) or str(uploaded_file)

    if not path:
        return None

    copied = copy_to_media(str(path))
    suffix = copied.suffix.lower()

    if suffix in IMAGE_EXTS:
        return image_to_video(str(copied))
    if suffix in AUDIO_EXTS:
        return audio_to_video(str(copied))
    if suffix in VIDEO_EXTS:
        return copied

    raise ValueError(
        f"Unsupported file type '{suffix}'. "
        "Upload video (.mp4/.mov/.avi/.webm), "
        "image (.png/.jpg/.webp), or audio (.wav/.mp3/.m4a)."
    )


# ── Analytics ───────────────────────────────────────────────────────────────
def robust_scale(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    lo, hi = np.nanpercentile(values, [2, 98])
    if not (np.isfinite(lo) and np.isfinite(hi) and hi > lo):
        return np.zeros_like(values)
    return np.clip((values - lo) / (hi - lo), 0.0, 1.0) * 100.0


def prediction_features(pred_array: np.ndarray) -> Dict[str, np.ndarray]:
    # Derive four NeuroAd proxy metrics from a TRIBE v2 array (T x V).
    arr  = np.asarray(pred_array, dtype=np.float32)
    T, V = arr.shape
    half = V // 2

    abs_peak      = np.nanpercentile(np.abs(arr), 95, axis=1)
    positive_mean = np.nanmean(np.maximum(arr, 0), axis=1)
    dispersion    = np.nanstd(arr, axis=1)
    delta = (
        np.r_[0.0, np.nanmean(np.abs(np.diff(arr, axis=0)), axis=1)]
        if T > 1 else np.zeros(T, dtype=np.float32)
    )
    lateral = (
        np.nanmean(arr[:, :half], axis=1) - np.nanmean(arr[:, half:], axis=1)
    )
    return {
        "attention": robust_scale(abs_peak),
        "memory":    robust_scale(positive_mean),
        "load":      robust_scale(0.65 * dispersion + 0.35 * delta),
        "valence":   robust_scale(lateral),
    }


def classify_frame(
    attention: float, memory: float, valence: float, load: float
) -> Tuple[str, str]:
    if load >= 78 and valence <= 40:
        return (
            "BAD - overload risk",
            "High processing effort with weak approach signal. "
            "Reduce clutter, simplify copy, or slow the edit.",
        )
    if attention >= 65 and memory >= 58 and valence >= 55 and load <= 72:
        return (
            "GOOD - strong creative moment",
            "Attention, memory, and approach are all elevated without excessive load.",
        )
    if attention < 35 and memory < 40:
        return (
            "WEAK - low salience",
            "The moment may need a stronger product reveal, "
            "motion cue, or audio/visual contrast.",
        )
    return (
        "NEUTRAL - usable",
        "No severe overload pattern, but not a standout attention or memory peak.",
    )


# ── Plotly helpers ──────────────────────────────────────────────────────────
def make_cortical_plot(pred_array: np.ndarray, t: int) -> go.Figure:
    row         = np.asarray(pred_array[int(t)], dtype=np.float32)
    left, right = np.array_split(row, 2)

    def _bin_mean(vals: np.ndarray, bins: int = 180) -> np.ndarray:
        return np.array(
            [np.nanmean(c) for c in np.array_split(vals, bins)],
            dtype=np.float32,
        )

    heat = np.vstack([_bin_mean(left), _bin_mean(right)])
    zmax = float(np.nanpercentile(np.abs(heat), 98)) or 1.0

    fig = go.Figure(data=go.Heatmap(
        z=heat, colorscale="Inferno", zmid=0, zmin=-zmax, zmax=zmax,
    ))
    fig.update_layout(
        title=f"Cortical activation strip — t = {int(t)} s",
        template="plotly_dark",
        height=320,
        margin=dict(l=30, r=30, t=50, b=35),
        yaxis=dict(
            tickmode="array", tickvals=[0, 1],
            ticktext=["Left hemi", "Right hemi"],
        ),
    )
    return fig


def make_timeline_plot(features: Dict[str, np.ndarray], t: int) -> go.Figure:
    x = np.arange(len(features["attention"]))
    _colors = {
        "attention": "#00d5ff",
        "memory":    "#ffcc00",
        "valence":   "#00e083",
        "load":      "#ff5a5f",
    }
    fig = go.Figure()
    for name, color in _colors.items():
        fig.add_trace(go.Scatter(
            x=x, y=features[name], mode="lines",
            name=name.title(), line=dict(color=color, width=3),
        ))
    fig.add_vline(x=int(t), line_width=2, line_dash="dash", line_color="white")
    fig.update_yaxes(range=[0, 100], title="Score (0-100)")
    fig.update_xaxes(title="Time (s)")
    fig.update_layout(
        template="plotly_dark", height=360,
        margin=dict(l=30, r=30, t=50, b=35),
    )
    return fig


def empty_plot(title: str) -> go.Figure:
    fig = go.Figure()
    fig.update_layout(title=title, template="plotly_dark", height=320)
    return fig


print("OK  NeuroAd helper functions loaded.")


OK  NeuroAd helper functions loaded.


In [7]:
# ── Cell 9: Build the NeuroAd Gradio dashboard ─────────────────────────────
# Requires Cells 1-8 to have completed successfully.

import traceback
import time
from typing import Dict, Optional, Any

import gradio as gr
import numpy as np
import pandas as pd

_SAMPLE_URL  = "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4"
_SAMPLE_PATH = CACHE_FOLDER / "sample_video.mp4"

_CSS = (
    ".gradio-container {"
    "  background: radial-gradient("
    "    circle at top left,"
    "    #142033 0%, #080a0f 42%, #05070a 100%"
    "  ) !important;"
    "}"
)


# ── Frame summary (shared by run and scrub callbacks) ──────────────────────
def _frame_summary(payload: Optional[Dict], t: int) -> str:
    if payload is None:
        return "Run a simulation to view analysis."
    arr  = payload["preds"]
    t    = int(np.clip(t, 0, len(arr) - 1))
    feat = prediction_features(arr)
    attn = float(feat["attention"][t])
    mem  = float(feat["memory"][t])
    val  = float(feat["valence"][t])
    ld   = float(feat["load"][t])
    label, advice = classify_frame(attn, mem, val, ld)
    lines = [
        f"### {label}",
        "",
        f"**Timestamp:** {t} s",
        "",
        f"**Attention:** {attn:.1f} / 100",
        f"**Valence:** {val:.1f} / 100",
        f"**Memory:** {mem:.1f} / 100",
        f"**Cognitive Load:** {ld:.1f} / 100",
        "",
        advice,
        "",
        "_NeuroAd proxy diagnostics derived from TRIBE v2 predicted BOLD "
        "activity. Calibrate with ROI masks and campaign outcome data "
        "before making production claims._",
    ]
    return "\n".join(lines)


# ── Main inference callback ─────────────────────────────────────────────────
def run_neuroad(uploaded_file, use_sample: bool):
    try:
        if "model" not in globals() or model is None:
            raise RuntimeError("TRIBE v2 model not loaded. Run Cell 5 first.")

        if use_sample:
            if not _SAMPLE_PATH.exists():
                download_file(_SAMPLE_URL, _SAMPLE_PATH)
            video_path = _SAMPLE_PATH
        else:
            video_path = normalize_upload(uploaded_file)

        if video_path is None:
            raise ValueError(
                "No input provided. "
                "Upload campaign media or tick 'Use Meta sample video'."
            )

        cache_key = file_digest(str(video_path))
        if cache_key in PREDICTION_CACHE:
            payload = PREDICTION_CACHE[cache_key]
        else:
            t0        = time.time()
            events_df = model.get_events_dataframe(video_path=video_path)
            pred_arr, segs = model.predict(events=events_df)
            pred_arr  = np.asarray(pred_arr, dtype=np.float32)
            payload   = {
                "video_path": str(video_path),
                "events":     events_df,
                "preds":      pred_arr,
                "segments":   segs,
                "runtime":    round(time.time() - t0, 2),
            }
            PREDICTION_CACHE[cache_key] = payload

        feat = prediction_features(payload["preds"])
        n    = len(payload["preds"])
        t    = 0

        status = (
            f"OK  Prediction complete  |  "
            f"Runtime: {payload['runtime']} s  |  "
            f"Frames: {n}  |  "
            f"Vertices: {payload['preds'].shape[1]}"
        )
        return (
            payload,
            status,
            gr.update(minimum=0, maximum=max(0, n - 1),
                      value=0, step=1, interactive=True),
            payload["video_path"],
            float(feat["attention"][t]),
            float(feat["valence"][t]),
            float(feat["memory"][t]),
            float(feat["load"][t]),
            make_cortical_plot(payload["preds"], t),
            make_timeline_plot(feat, t),
            _frame_summary(payload, t),
            payload["events"].head(80),
        )

    except Exception as exc:
        print(traceback.format_exc())
        err_md = f"**{type(exc).__name__}:** {exc}"
        return (
            None,
            f"Run failed - {type(exc).__name__}: {exc}",
            gr.update(minimum=0, maximum=1, value=0, step=1, interactive=False),
            None,
            0.0, 0.0, 0.0, 0.0,
            empty_plot("Cortical activation - unavailable"),
            empty_plot("Telemetry timeline - unavailable"),
            err_md,
            pd.DataFrame(),
        )


# ── Frame-scrub callback ────────────────────────────────────────────────────
def update_frame(payload, timestamp):
    if payload is None:
        return (
            0.0, 0.0, 0.0, 0.0,
            empty_plot("Cortical activation - no data"),
            empty_plot("Timeline - no data"),
            "Run a simulation first.",
        )
    arr  = payload["preds"]
    t    = int(np.clip(timestamp, 0, len(arr) - 1))
    feat = prediction_features(arr)
    return (
        float(feat["attention"][t]),
        float(feat["valence"][t]),
        float(feat["memory"][t]),
        float(feat["load"][t]),
        make_cortical_plot(arr, t),
        make_timeline_plot(feat, t),
        _frame_summary(payload, t),
    )


# ── UI layout ───────────────────────────────────────────────────────────────
with gr.Blocks(css=_CSS, theme=gr.themes.Base()) as neuroad_app:
    state = gr.State(None)

    gr.Markdown("# NeuroAd TRIBE v2 - Campaign Brain Lab")
    gr.Markdown(
        "Upload campaign media, run Meta's TRIBE v2 brain encoding model, "
        "and inspect frame-level NeuroAd diagnostics."
    )

    with gr.Row():
        with gr.Column(scale=1):
            upload = gr.File(
                label="Campaign asset (video / image / audio)",
                file_types=["video/*", "image/*", "audio/*"],
            )
            use_sample = gr.Checkbox(
                label="Use Meta sample video (Sintel trailer)", value=False
            )
            run_btn = gr.Button("Run Brain Simulation", variant="primary")
            status  = gr.Textbox(
                label="Status", value="Ready.", lines=3, interactive=False
            )
            video_preview = gr.Video(label="Stimulus preview")

        with gr.Column(scale=2):
            timeline = gr.Slider(
                minimum=0, maximum=1, value=0, step=1,
                label="Playback timestamp (seconds)",
                interactive=False,
            )
            cortex_plot    = gr.Plot(label="Cortical activation")
            telemetry_plot = gr.Plot(label="NeuroAd telemetry timeline")

    with gr.Row():
        with gr.Column(scale=1):
            attn_sl = gr.Slider(0, 100, value=0,
                                label="Attention Capture",  interactive=False)
            val_sl  = gr.Slider(0, 100, value=0,
                                label="Emotional Valence",  interactive=False)
            mem_sl  = gr.Slider(0, 100, value=0,
                                label="Memory Encoding",    interactive=False)
            load_sl = gr.Slider(0, 100, value=0,
                                label="Cognitive Load",     interactive=False)
        with gr.Column(scale=2):
            summary = gr.Markdown("Run a simulation to view analysis.")

    with gr.Accordion("Events dataframe preview", open=False):
        events_table = gr.Dataframe(label="Events", interactive=False)

    run_btn.click(
        run_neuroad,
        inputs=[upload, use_sample],
        outputs=[
            state, status, timeline, video_preview,
            attn_sl, val_sl, mem_sl, load_sl,
            cortex_plot, telemetry_plot, summary, events_table,
        ],
    )
    timeline.change(
        update_frame,
        inputs=[state, timeline],
        outputs=[
            attn_sl, val_sl, mem_sl, load_sl,
            cortex_plot, telemetry_plot, summary,
        ],
    )

print("OK  Dashboard built. Run Cell 10 to launch.")


OK  Dashboard built. Run Cell 10 to launch.


In [8]:
# Cell 10: Launch the NeuroAd dashboard - Kaggle-safe Gradio launch

try:
    neuroad_app.close()
except Exception:
    pass

# Kaggle/Gradio workaround:
# Gradio's API schema route crashes on some package combos with:
# TypeError: argument of type 'bool' is not iterable
# The UI does not need this API schema, so we return an empty one.
try:
    import gradio.blocks as _gr_blocks

    if not getattr(_gr_blocks.Blocks, "_neuroad_api_info_patch", False):
        def _safe_get_api_info(self):
            return {
                "named_endpoints": {},
                "unnamed_endpoints": {},
            }

        _gr_blocks.Blocks.get_api_info = _safe_get_api_info
        _gr_blocks.Blocks._neuroad_api_info_patch = True
        print("Applied Gradio API-info compatibility patch.")
except Exception as exc:
    print("Gradio API-info patch skipped:", exc)

neuroad_app.queue(max_size=4)

app, local_url, share_url = neuroad_app.launch(
    share=True,
    debug=False,
    show_error=True,
    prevent_thread_lock=True,
    quiet=False,
)

print("Local URL:", local_url)
print("Public URL:", share_url)
print("Open the Public URL from THIS run.")

Applied Gradio API-info compatibility patch.
Running on local URL:  http://127.0.0.1:7860
Running on public URL: https://d1992c02dcfe799285.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Local URL: http://127.0.0.1:7860/
Public URL: https://d1992c02dcfe799285.gradio.live
Open the Public URL from THIS run.


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 3281.93it/s]
/usr/local/lib/python3.12/dist-packages/neuralset/events/transforms/audio.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([events, pd.DataFrame(events_to_add)], ignore_index=True)
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)
Extracting words from audio: 100%|██████████| 1/1 [00:00<00:00, 434.19it/s]
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This 

In [10]:
import shutil
from pathlib import Path

src = Path("/content/neuroad_exports")
dst = Path("/kaggle/working/neuroad_exports")

dst.mkdir(parents=True, exist_ok=True)

for file in src.glob("*"):
    shutil.copy2(file, dst / file.name)

print("Copied exports to:", dst)
print("Files:")
for file in dst.glob("*"):
    print(" -", file)

Copied exports to: /kaggle/working/neuroad_exports
Files:
 - /kaggle/working/neuroad_exports/prediction_9b678890fb8ca401.npy
 - /kaggle/working/neuroad_exports/events_9b678890fb8ca401.csv


In [11]:
import shutil
from IPython.display import FileLink, display

zip_path = shutil.make_archive(
    "/kaggle/working/neuroad_exports",
    "zip",
    "/kaggle/working/neuroad_exports"
)

display(FileLink(zip_path))
print(zip_path)

/kaggle/working/neuroad_exports.zip

/kaggle/working/neuroad_exports.zip


In [14]:
from pathlib import Path
import shutil
import os

src = Path("/kaggle/working/neuroad_exports")
zip_path = Path("/kaggle/working/neuroad_exports.zip")

if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_path.with_suffix("")), "zip", src)

print("Created zip:", zip_path)
print("Size:", zip_path.stat().st_size, "bytes")
print("Files in /kaggle/working:")
for f in Path("/kaggle/working").iterdir():
    print(" -", f.name)

Created zip: /kaggle/working/neuroad_exports.zip
Size: 4006901 bytes
Files in /kaggle/working:
 - neuroad_exports
 - .virtual_documents
 - neuroad_exports.zip
 - neuroad_exports_download.zip
